# MLB Offensive Production Analysis

This notebook evaluates same-season associations between offensive production and hitter traits for model-eligible player-seasons in the cached 2024–2026 dataset. It is not a causal analysis or a next-season forecast.

Primary outcome: FanGraphs **wRC+**. OPS, OPS+, wOBA, and xwOBA are reported as outcomes/context but are intentionally excluded from wRC+ predictors to avoid formulaic leakage. Evaluation uses grouped cross-validation by `player_id`, so a player's repeated seasons are never split between training and test data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from mlb_offense.analysis import (
    AnalysisConfig,
    bootstrap_model_difference,
    clustered_ols_coefficients,
    coverage_by_season,
    evaluate_models,
    feature_sets,
    grouped_permutation_importance,
    load_analysis_data,
    missingness_by_season,
    plot_correlation,
    plot_missingness,
    plot_outcome_distributions,
    plot_predictions,
    plot_residual_diagnostics,
)

DATA_PATH = Path('data/processed/mlb_offense_2024_2026.parquet')
OUTPUT_DIR = Path('data/analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = AnalysisConfig(
    min_pa=100,
    min_competitive_swings=50,
    outer_splits=5,
    inner_splits=3,
    bootstrap_iterations=500,
)
data = load_analysis_data(DATA_PATH, config)
blocks = feature_sets(data)
print(f'{len(data):,} eligible player-seasons across {data.player_id.nunique():,} hitters')

## Sample coverage and descriptive outcomes

The local thresholds are intentionally applied after data collection. This keeps the raw source responses intact and makes sensitivity checks straightforward.

In [ ]:
display(coverage_by_season(data).style.format(precision=3))
display(
    data.groupby('season')[['pa', 'competitive_swings', 'wrc_plus', 'ops_plus', 'woba', 'xwoba']]
    .describe()
)

In [ ]:
figures = [
    plot_outcome_distributions(data),
    plot_missingness(data),
    plot_correlation(data, blocks['core_plus_traits']),
]
for number, figure in enumerate(figures, start=1):
    figure.savefig(OUTPUT_DIR / f'eda_{number}.png', dpi=160, bbox_inches='tight')
    plt.show()

display(missingness_by_season(data).sort_values(['season', 'missing_rate'], ascending=[True, False]).head(20))

## Leakage-safe grouped model evaluation

Predictors are evaluated in two prespecified blocks: `core` contains age, handedness, season, plate discipline, and contact-quality measures; `core_plus_traits` adds bat-tracking and stance fields. Production metrics and their components are excluded. Hyperparameters are tuned inside each training fold; the outer evaluation folds are grouped by hitter.

In [ ]:
performance, predictions, selected_parameters = evaluate_models(data, config)
performance.to_csv(OUTPUT_DIR / 'grouped_cv_performance.csv', index=False)
predictions.to_parquet(OUTPUT_DIR / 'out_of_fold_predictions.parquet', index=False)
selected_parameters.to_json(OUTPUT_DIR / 'selected_parameters.json', orient='records', indent=2)

display(
    performance.style.format({
        'mae': '{:.2f}', 'rmse': '{:.2f}', 'r2': '{:.3f}',
        'calibration_intercept': '{:.2f}', 'calibration_slope': '{:.3f}',
        'pa_weighted_mae': '{:.2f}', 'pa_weighted_rmse': '{:.2f}',
        'pa_weighted_r2': '{:.3f}',
    })
)

In [ ]:
comparison = bootstrap_model_difference(
    predictions,
    candidate_model='elastic_net:core_plus_traits',
    reference_model='elastic_net:core',
    metric='mae',
    iterations=config.bootstrap_iterations,
    random_state=config.random_state,
)
comparison.to_csv(OUTPUT_DIR / 'elastic_net_trait_increment_bootstrap.csv', index=False)
display(comparison.style.format({'difference': '{:.3f}', 'ci_lower': '{:.3f}', 'ci_upper': '{:.3f}'}))

best_model = performance.iloc[0]['model']
prediction_figure = plot_predictions(predictions, best_model)
prediction_figure.savefig(OUTPUT_DIR / 'best_model_calibration.png', dpi=160, bbox_inches='tight')
plt.show()

residual_figure = plot_residual_diagnostics(predictions, best_model, data)
residual_figure.savefig(OUTPUT_DIR / 'best_model_residuals.png', dpi=160, bbox_inches='tight')
plt.show()

## Interpretation aids

The standardized OLS coefficients below use player-clustered standard errors. Permutation importance is descriptive: correlated hitter traits can substitute for each other and are not independent causal effects.

In [ ]:
coefficients = clustered_ols_coefficients(data, blocks['core_plus_traits'], config)
coefficients.to_csv(OUTPUT_DIR / 'clustered_ols_coefficients.csv', index=False)
display(coefficients.head(20).style.format({
    'coefficient': '{:.3f}', 'std_error': '{:.3f}', 'ci_lower': '{:.3f}',
    'ci_upper': '{:.3f}', 'p_value': '{:.4f}',
}))

importance = grouped_permutation_importance(data, blocks['core_plus_traits'], config)
importance.to_csv(OUTPUT_DIR / 'permutation_importance.csv', index=False)
display(importance.head(20).style.format({'importance_mean': '{:.3f}', 'importance_std': '{:.3f}'}))

## Interpretation checklist

- The table compares observed production in the same season, so it should be interpreted as association rather than prediction or causation.
- Compare `elastic_net:core_plus_traits` against `elastic_net:core` and inspect the bootstrapped MAE difference to assess incremental value from bat and stance traits.
- Check residual plots and calibration before trusting aggregate error statistics.
- Re-run with alternative PA and competitive-swing thresholds before making conclusions about the MLB hitter population.